# Module 2: Units and the Memory Budget

In Module 1 you reached a server you own and traced one request. Before you open the box in Module 3, you need the units the rest of the workshop counts in. This module is short and runs anywhere, no GPU needed. You count parameters and bytes, turn a model size into gigabytes, estimate tokens, and split your card's memory into weights, the KV cache, and overhead. By the end you can size a model against a card in your head.

## Learning objectives
- Define a parameter and a byte, and read precision as bytes per number
- Turn a model's parameter count into a memory footprint at FP16, FP8, and INT4
- State what a token is, and why your server caps a request at 2048 tokens by default
- Name the two GPU numbers that matter, VRAM and memory bandwidth
- Split your card's memory into weights and KV cache, and see how the under-tuned 0.7 default limits concurrency
- Watch a single agent's context grow to the 2048 cap, and read the budget as concurrent agent sessions

## Prerequisites
- Finished Module 1
- No GPU and no server needed, the whole module is arithmetic
- About 6 minutes

References: [vLLM](https://docs.vllm.ai) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [Qwen3-4B model card](https://huggingface.co/Qwen/Qwen3-4B) &middot; [vLLM engine args](https://docs.vllm.ai/en/latest/serving/engine_args.html)

## Memory design basics

A model is a pile of numbers, and a GPU is a fixed amount of fast memory. Most of what you tune later is the fit between the two.

- A **parameter** is one number the model learned. A 4B model has 4 billion of them.
- **Precision** is how many bytes one parameter takes. FP16 and BF16 are 2 bytes, FP8 is 1, INT4 is half a byte.
- **Model size** is parameters times bytes. Your served model, `RedHatAI/Qwen3-4B-FP8-dynamic`, is FP8 and measures about 6 GB.
- **VRAM** is how much the card holds, 20 GB here. **Bandwidth** is how fast it reads, 360 GB/s.
- The card's memory splits into the weights and the KV cache. The weights are fixed. The KV cache is the part you trade for more users, and your server starts with it choked down.

![The 20 GB card split into weights and KV cache, at the under-tuned 0.7 default versus the 0.9 tune](images/02_units_and_memory_budget_architecture.png)

## 1. Setup

No server and no GPU. This module is arithmetic, so the only dependency is the plotting library for the budget chart. We reinstall here so this notebook stands on its own.

In [ ]:
%pip install -q matplotlib

## 2. Parameters, bytes, and precision

A parameter is a number, and precision is how many bytes you spend to store it. Fewer bytes per number is the idea behind quantization, which Omer takes to real models in Module 5. Read the bytes per number for each precision.

In [ ]:
# Bytes per parameter at each precision. Fewer bytes is the basis of quantization (Module 5).
bytes_per_param = {"FP32": 4, "FP16": 2, "BF16": 2, "FP8": 1, "INT4": 0.5}
for name, b in bytes_per_param.items():
    print(f"{name:5}: {b} bytes per parameter")

**What you should see:** four bytes down to half a byte. FP16 is the common default at 2 bytes, and your served model is FP8 at 1. Module 5 spends fewer and measures what it costs in accuracy.

## 3. Model size from parameters

Model size is one multiplication: parameters times bytes per parameter. This is the first number you run before you pick a card, because the weights have to fit with room left over.

In [ ]:
# Model size = parameters * bytes per parameter.
params = 4e9   # a 4B model
for prec in ("FP16", "FP8", "INT4"):
    gb = params * bytes_per_param[prec] / 1e9
    print(f"4B at {prec:4}: {gb:.1f} GB of weights")

**What you should see:** about 8 GB at FP16, 4 GB at pure FP8, 2 GB at INT4. Your served model is FP8-dynamic, which keeps a few layers (embeddings, the output head) in higher precision, so it measures about 6 GB, between the two. Use the measured number. That freed memory becomes KV cache, which is section 6.

## 4. What a token is

A token is not a word. A tokenizer splits text into pieces of about four characters, so you can estimate token counts in your head. Tokens are the unit a provider bills, the unit the model emits one at a time, and the unit the KV cache stores.

In [ ]:
# A token is about 4 characters of English. The tokenizer is exact; this is the head estimate.
sample = "Owning your inference means owning the layer under your agent."
est_tokens = len(sample) // 4
print(f"text: {sample!r}")
print(f"characters: {len(sample)}  ->  about {est_tokens} tokens (4 chars each)")

**What you should see:** a token estimate near a quarter of the character count. One thing to hold onto: your server caps prompt plus generation at 2048 tokens by default (`--max-model-len=2048`), so keep examples within it. Raising that cap is part of the Module 9 tune.

## 5. VRAM and bandwidth

Two numbers describe your card. VRAM is how much fits. Bandwidth is how fast the GPU reads it. They map to the two phases from Module 1: prefill is limited by compute, and decode is limited by bandwidth, because each token reloads the whole weight set out of memory. That gives a single-request speed limit you can compute now.

In [ ]:
# The two card numbers, and the decode speed limit they imply.
card_vram_gb = 20.0      # RTX 4000 Ada, from nvidia-smi
bandwidth_gbs = 360.0    # from the card datasheet
served_weights_gb = 6.0  # RedHatAI/Qwen3-4B-FP8-dynamic, measured (FP16 would be 8)

# Decode reloads every weight to make one token, so one request goes no faster than bandwidth / weights.
decode_ceiling = bandwidth_gbs / served_weights_gb
print(f"VRAM      : {card_vram_gb:.0f} GB")
print(f"bandwidth : {bandwidth_gbs:.0f} GB/s")
print(f"single-request decode ceiling: about {decode_ceiling:.0f} tokens/s")

**What you should see:** a decode ceiling near 60 tokens per second for the served FP8 4B on a 360 GB/s card. A smaller or more quantized model has a higher floor: the 0.6B sits several times above this. No software makes one request faster than its floor. Module 4 draws this as a plot, and Module 3 measures it on your own GPU.

## 6. The memory budget

The card's VRAM is a budget. The weights take a fixed slice, and what is left is KV cache, the part that decides how many users fit. Your server starts under-tuned at `--gpu-memory-utilization=0.7`, so the cache is small. Raising it is the Module 9 tune. Split the budget at 0.7 and at 0.9 and see the difference.

In [ ]:
# Split the 20 GB card into weights and KV cache, under-tuned 0.7 versus tuned 0.9.
for util in (0.70, 0.90):
    kv = util * card_vram_gb - served_weights_gb
    tag = "default, under-tuned" if util == 0.70 else "tuned in Module 9"
    print(f"util {util}: weights {served_weights_gb:.0f} GB  ->  KV cache {kv:.0f} GB   ({tag})")

**What you should see:** about 8 GB of KV cache at the under-tuned 0.7 default, and about 12 GB once Module 9 raises utilization to 0.9. Same card, half again as much room for users, from one flag. (Overhead eats another 1 to 2 GB in practice.) Module 3 turns that KV slice into a per-token cost and a concurrency ceiling.

In [ ]:
# A stacked bar of the budget, under-tuned 0.7 versus tuned 0.9.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

labels = ["0.7 default", "0.9 tuned"]
utils = [0.70, 0.90]
weights = [served_weights_gb, served_weights_gb]
kv = [u * card_vram_gb - served_weights_gb for u in utils]

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.bar(labels, weights, label="weights")
ax.bar(labels, kv, bottom=weights, label="KV cache")
ax.set_ylabel("GB of the 20 GB card")
ax.set_title("Raising gpu-memory-utilization grows the KV cache")
ax.legend()
fig.tight_layout()

**What you should see:** two bars, the KV slice taller at 0.9. That is the tune you make in Module 9, and the picture Module 3's concurrency ceiling builds on.

Read the budget as agent capacity, not just bytes. Divide the KV slice by one agent's context and you get how many concurrent agent sessions fit on your card. A short-context chat fits many. A long-context agent, carrying a system prompt, tool schemas, and a growing scratchpad, fits few. Doubling each agent's context roughly halves how many fit. That trade is the economics of hosting agents on a GPU you own, and it is why the next section watches a single agent eat the budget.

## 7. The agent context tax

A chatbot sends one prompt. An agent loops, and every turn it appends its Thought, its Action, and the tool's Observation, then resends the whole thing. Context grows every turn and never shrinks. With a system prompt and tool schemas around 700 tokens and 300-token observations, you reach the 2048 cap in a few turns. Run the loop from `common/agent_loop.py` against your own server and watch it climb.

In [ ]:
# Requires a live vLLM endpoint. Run a ReAct loop and watch the prompt grow toward the cap.
from common import agent_loop
records = agent_loop.run(steps=8, obs_tokens=300, max_tokens=120)
for r in records:
    if r["ok"]:
        print(f"step {r['step']}: prompt_tokens={r['prompt_tokens']:>5}  KV={r['prompt_tokens'] * 144 / 1024:.0f} MiB")
    else:
        print(f"step {r['step']}: FAILED  {r['error'][:90]}")

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
ok = [r for r in records if r["ok"] and r["prompt_tokens"]]
plt.figure(figsize=(7, 3))
plt.plot([r["step"] for r in ok], [r["prompt_tokens"] for r in ok], "o-", label="prompt tokens")
plt.axhline(2048, ls="--", color="red", label="--max-model-len cap")
plt.xlabel("agent step"); plt.ylabel("prompt tokens"); plt.legend(); plt.tight_layout()

**What you should see:** the prompt climbs every turn and crosses the 2048 line in a few steps, then the server rejects the next request with a context-length error. Each turn also adds about 300 times 144 KiB of KV. The 2048 cap is a task-length ceiling for your agent, not arbitrary austerity. Raising `--max-model-len` (the Module 9 tune) buys more turns, at the cost of more KV cache reserved per request, which lowers how many agents fit.

## Things to know

- **Model size is the first number you run.** Parameters times bytes. The cache, the concurrency, and the cost all start from how much the weights take.
- **Fewer bytes per weight buys two things.** A quantized model reads fewer bytes per token, so it decodes faster, and it frees memory that becomes KV cache, so more users fit. Your FP8 model already shows the first half: it decodes faster than the same model at FP16. Omer measures the accuracy cost in Module 5.
- **Your context is 2048 tokens by default.** `--max-model-len=2048` caps prompt plus generation. Author examples inside it. To fill the KV cache you raise concurrency, not prompt length, until Module 9 raises the cap.
- **The models reason first.** These are Qwen3 thinking models, so `message.content` is empty unless you give enough tokens or turn thinking off. The workshop's `build_client` turns it off for measurement, so your answers come back clean.

## Try it yourself

**Size a bigger model.** Change `your_params` to 8e9, then 14e9, at each precision, and find where the model stops leaving room on your 20 GB card. **Stretch:** the 30B MoE in Module 4 has 30B total parameters. Confirm it does not fit, even at INT4 with no cache.

**Move the budget.** Change `util` to 0.95 and watch the KV slice grow. That single flag is the lever Du'An raises in Module 9.

In [ ]:
# Change these, then run the cell.
your_params = 8e9          # try 8e9, 14e9, 30e9
your_precision = "FP8"     # try FP16, INT4
util = 0.90

w = your_params * bytes_per_param[your_precision] / 1e9
kv = util * card_vram_gb - w
fits = f"fits, {kv:.0f} GB left for cache" if kv > 0 else "does NOT fit"
print(f"{your_params/1e9:.0f}B at {your_precision}: weights {w:.1f} GB -> {fits}")

## Summary

- A parameter is a number, and precision is its size in bytes. FP16 is 2 bytes, FP8 is 1, INT4 is half.
- Model size is parameters times bytes. Your served FP8 4B measures about 6 GB.
- A token is about four characters, and your server caps a request at 2048 of them by default.
- The card's VRAM splits into weights and KV cache. The under-tuned 0.7 default leaves about 8 GB of cache; the 0.9 tune in Module 9 leaves about 12 GB.

## Next

**Module 3: Prefill, Decode, and the KV Cache.** You can size a model now. Next you build attention and the KV cache by hand, watch generation go from quadratic to linear, and measure the exact per-token cache cost on your own GPU.